# W10B · 실습 B · 실제 MovieLens와사용자/아이템AE

**딥러닝응용I(추천시스템) · 동덕여자대학교 데이터사이언스전공 · 유원상 교수 · 2026-2 · 2026-11-05**

공식MovieLens 100K를자동준비하고관측분할·훈련 평균·소형CPU학습·최종평가를연결합니다. 방향을바꾸어주교재연습7-1을수행합니다.

Colab에서 자신의 사본을 저장하고 첫 셀부터 실행하세요. 설치 후 재시작 안내가 나오면 런타임을 재시작한 뒤 다시 처음부터 실행합니다. 개인 파일 업로드·API키·유료GPU가 필요하지 않습니다. 이 노트북은 독립적으로 입력을 준비합니다.

`None`은 자신의 답을 적는 자리이며 기본 계산 변수와 분리되어 있습니다. 빈 응답이 있어도 예제 실행은 계속되지만,이는 연습을 완료했다는 뜻이 아닙니다. 예상 → 직접 계산 → 실행 → 이유 한 문장 순서로 기록하세요.


In [ ]:
import os, sys, subprocess
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
os.environ['GRADIO_ANALYTICS_ENABLED'] = 'False'
IN_COLAB = 'google.colab' in sys.modules
COURSE_VERSION = '2026-fall-w10b-r1'
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
        f'luna-recommender-course[apps,neural] @ git+https://github.com/lunalab-ai/recommender.git@{COURSE_VERSION}',
        'gradio==6.31.0'], check=True)
    for package, wanted in [('tensorflow','2.21.0'), ('keras','3.15.1')]:
        if package in sys.modules and getattr(sys.modules[package], '__version__', None) != wanted:
            raise RuntimeError('설치가 갱신되었습니다. 런타임을 재시작한 뒤 첫 셀부터 다시 실행하세요.')
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
import keras
from IPython.display import display
from luna_recsys.autoencoder_recommenders import (
    toy_ratings, fixed_ae_forward, masked_metrics, loss_comparison,
    build_fixed_autoencoder, prepare_autoencoder_data, oriented_arrays,
    fit_autoencoder, predict_autoencoder, evaluate_autoencoder, recommend_autoencoder,
)
from luna_recsys.autoencoder_explainer import (
    ae_view, export_ae_csv, build_autoencoder_explainer,
)
from luna_recsys.datasets import prepare_movielens
print('Python',sys.version.split()[0], 'NumPy',np.__version__, 'TensorFlow',tf.__version__, 'Keras',keras.__version__)
print('수업 ref:',COURSE_VERSION,'CPU 실행; 학습 자료 모드를 결과와 함께 확인합니다.')


## 실제 자료를자동준비하기

`mode='real'`은공식자료의다운로드/캐시를사용하며실패를합성자료로바꾸지않습니다. 분할과평균은`prepare_autoencoder_data`가준비합니다. `values['train']`만모델 입력으로사용합니다.

<img src="https://raw.githubusercontent.com/lunalab-ai/recommender/2026-fall-w10b-r1/course/notion/assets/w10b/10-data-split.png" alt="훈련·검증·시험관측의역할" width="1100">

[그림 원본](https://raw.githubusercontent.com/lunalab-ai/recommender/2026-fall-w10b-r1/course/notion/assets/w10b/10-data-split.png)

그림은작은독자8/2/2설명예이고,이노트북의실제분할은70/15/15입니다.


In [ ]:
prepared = prepare_movielens(Path.home()/'.cache/luna-recsys',mode='real')
data = prepare_autoencoder_data(prepared,seed=20261105)
print('자료 모드:',prepared.mode,prepared.description)
print('사용자×영화:',data.shape,'훈련 평균:',data.mean)
counts = {name:int(mask.sum()) for name,mask in data.masks.items()}
print('관측 수:',counts)
baseline = evaluate_autoencoder(data,np.full(data.shape,data.mean),split='test')
print('최종시험의평균기준 모형:',baseline)


## B01 · 입력과 평가 정답 구별

평균을훈련관측에서만계산하는이유를적으세요. 검증/시험관측위치에모델 입력이어떤값인지확인하고,그0이정답평점0을뜻하는지설명하세요.


In [ ]:
my_training_mean_reason = None
my_input_target_explanation = None


## 작은실제모형학습과검증기준선택

`DIRECTION`을정한뒤실행합니다. 기본은사용자기준입니다. 은닉폭32/8/32,L1=.0003,Adamax학습률.001,최대8epoch입니다. 관측 MSE+L1을학습하고검증 RMSE최상가중치를복원합니다. 교재원형의전체 대형 학습을재현하는설정이아닙니다.


In [ ]:
DIRECTION = 'user'
values, masks = oriented_arrays(data,DIRECTION)
print('선택한 입력 shape:',values['train'].shape)
fitted = fit_autoencoder(data,direction=DIRECTION,epochs=8,hidden=(32,8),reg=.0003,
                         learning_rate=.001,batch_size=128,seed=20261105)
display(fitted.history)
print('선택 epoch:',fitted.best_epoch,'모수:',fitted.model.count_params(),
      '갱신수:',fitted.updates,'훈련/평가초:',fitted.train_seconds)
prediction = predict_autoencoder(fitted,data)
selected_metric = evaluate_autoencoder(data,prediction,split='test')
display(pd.DataFrame([{'model':'mean',**baseline},{'model':DIRECTION+' AE',**selected_metric}]))


## B02 · 모형과설정값읽기

모델 입력폭과 출력폭,병목크기를확인하세요. L1을어느층에적용했으며,학습 목적과보고RMSE가왜다른지설명하세요.


In [ ]:
my_model_shape = None
my_regularization_explanation = None


## L07 · loss·배치집계·선택기준

워크북L07의가중치/L1=.01/MSE=.64를계산하세요. 이어관측 수2/8과SSE2/32를합쳐전체RMSE를구하고검증기준의epoch선택을설명하세요.


In [ ]:
my_l1_loss = None
my_total_objective = None
my_global_rmse = None
my_epoch_choice = None


## B03 / L08 · 주교재연습7-1의비교

같은data를유지하고DIRECTION을바꾸어실행하세요. 입력 shape·관측 수·평균·설정·모수 수·갱신수·시간·검증선택 epoch·시험 RMSE를기록합니다. 샘플하나의의미가어떻게달라졌나요?


In [ ]:
my_other_direction = None
my_comparison_record = None
my_comparison_explanation = None


## 알고있는이력을제외한후보와실제앱

아래후보는사용자1의알려진관측이력을제외한표입니다. RMSE실험과추천목록의품질검증을구별하세요. 앱의 고정 계산과합성학습탭은실제 100K성능실험과자료 모드가다릅니다.


In [ ]:
recommendations = recommend_autoencoder(data,prediction,1,k=5)
display(recommendations)
demo = build_autoencoder_explainer()
if IN_COLAB:
    demo.queue(default_concurrency_limit=1).launch(share=True,inline=False,prevent_thread_lock=True)
else:
    print('앱UI실행은별도검증; 여기서는객체와계산콜백을확인합니다.')


## B04 · 추천목록의해석

알려진이력이목록에없는지확인하세요. 시험 RMSE가낮다는이유만으로상위추천의만족도를검증했다고쓸수있나요?


In [ ]:
my_ranking_quality_explanation = None


## B05 · 공정한비교와재실행

두방향의epoch가같으면계산예산도같은가요? 설정을바꾸어반복할때검증과 시험을어떻게사용해야할지적으세요.


In [ ]:
my_compute_budget_explanation = None
my_repeated_selection_rule = None


In [ ]:
assert prepared.mode in {'movielens','local'}
assert data.shape==(943,1682) and counts=={'train':70000,'validation':15000,'test':15000}
assert sum(mask.astype(int) for mask in data.masks.values()).max()==1
assert np.isfinite(prediction).all() and prediction.shape==data.shape
assert selected_metric['n']==15000 and np.isfinite(selected_metric['rmse'])
assert data.values['train'][data.masks['validation']|data.masks['test']].sum()==0
assert np.isclose(selected_metric['rmse']**2,selected_metric['sse']/selected_metric['n'])
assert np.isfinite(fitted.history.objective).all()
assert ae_view()[1].shape[0]==4
print('W10B_B_RUNTIME_PASS actual MovieLens model checks; learner responses are not completion claims')


## 점검 퀴즈

1. 사용자 기준 AE의 한 입력 샘플은 무엇이며, MF의 ID 입력과 어떻게 다른가요?
2. 입력이 4차원이고 잠재표현이 2차원일 때, 출력이 다시 4차원이어야 하는 이유는 무엇인가요?
3. 입력 `[2,0,0,−2]`와 E02의 가중치에서 두 잠재 노드의 가중합과 활성값은 각각 무엇인가요?
4. 원래 평점3, 훈련 평균3인 관측이 중심화 후 0이 되면, 오차 계산에 포함해야 하나요?
5. E04의 관측 제곱오차 합 6과 관측 수 3에서 MSE와 RMSE는 각각 얼마인가요?
6. 관측 MSE 2와 L1 벌점 .0105가 주어졌을 때, 최적화 목적함수와 순수 RMSE는 어떻게 다른가요?
7. 교재의 `validation_data=(train_data, test_data)`에서 모델 입력과 비교할 정답은 각각 무엇인가요?
8. 같은 관측 분할로 사용자 기준과 아이템 기준을 비교할 때, RMSE 외에 어떤 조건과 수를 함께 기록해야 하나요?


## 기억할 것과 참고자료

입력과 관측 여부, 순전파와 학습, 검증과 시험을 구별하세요. 함수의 배열 크기와 상태 변경은 API 길잡이에서 확인합니다. 예상·수정·실행·이유를 자신의 말로 기록하세요.

- 주교재 임일(2025), 7장 전체, pp.148–156, 연습 7-1.
- [강의노트와워크북](https://github.com/lunalab-ai/recommender/tree/2026-fall-w10b-r1/course/notion)
- [Keras Dense](https://keras.io/api/layers/core_layers/dense/)
- [Keras Masking](https://keras.io/api/layers/core_layers/masking/)
- [AutoRec 원논문](https://ssanner.github.io/papers/anu/www15.pdf)
- [MovieLens 이용조건](https://files.grouplens.org/datasets/movielens/ml-100k-README.txt)
